# import

In [2]:
import pandas as pd
import random
from datetime import datetime, timezone

from utils.train_evaluate import setup_config_and_dataset
from utils.plot_utils import *
from utils.data_utils import get_id2token, recbole_ds_column_2_dataframe, recbole_dataset_2_external_id_df, save_dataframe_2_atomic_file

# variables

In [3]:
freq=12 # month
duration = 2*12//freq # 2 years split in xM buckets
n_parts = duration*2+1
d_keys = ['_pt'+str(i) for i in range(1, n_parts)]
# MODEL_VERSIONS = ['_pt1', '_pt2']
MODEL_VERSIONS = d_keys[:duration]


Ks = [1, 10, 20]
VM_K = Ks[2] # valid metric k, also used in heatmap matrix
VALID_METRIC = 'Recall@'+str(VM_K)
SEED = 2020
USE_GPU = False
SHOW_PROGRESS = False
SAVE_DATASET = False

# these are the default values
# TRAIN_NEG_SAMPLE_ARGS = {'distribution': 'uniform', 
#                          'sample_num': 1, 
#                          'alpha': 1.0, 
#                          'dynamic': False, 
#                          'candidate_num': 0}



SHUFFLE = False  # shuffle (bool): Whether or not to shuffle the training data before each epoch. Defaults to True.
EVAL_ARGS = {'split': {'LS': 'test_only'}, # leave-one-out sample type ['valid_and_test', 'valid_only', 'test_only']
                    'group_by': 'user',
                    'order': 'TO', # order (str): decides how we sort the data in .inter. random ordering or time ordering
                    'mode': 'uni100'}

METRICS = ['Recall', 'MRR', 'NDCG', 'Hit', 'Precision', 'GiniIndex', 'TailPercentage']

# FILENAME_VERSION = '_ET_ND_LS.t_UD_SF_TO_UM.100'

data_types_dict = {'item_id':'object', # bc of drifted items, 'd_xxxxx'
                   'user_id':'object',
                   'timestamp':'float32'}

# read Pre train

In [4]:
save_path, base_filename, _ = ('processed_datasets/natural_data/palco2010/two_intervals/',
                               'more_2interQ_df', 
                               '') 

pretrain_dataset_name = base_filename+'_PT'
pretrain_dir = save_path+pretrain_dataset_name+'/'+pretrain_dataset_name+'.csv'
pretrain = pd.read_csv(pretrain_dir)

# Random Drift - 50% of items

## load pt2, save pt2's valid and test

In [5]:
model_name = 'BPR' # for the sake of having one, BPR was chosen, but any other in theory yields the same results


save_path, base_filename, specs_str = ('processed_datasets/natural_data/palco2010/two_intervals/',
                                        'more_2interQ_df', 
                                        'NPT_RD.50') 
save_specs_str =  'PT_RD.50'
BENCHMARK_FILENAMES = ['train', 'valid', 'test']

base_dataset_name = base_filename+'_'+specs_str


# from complete dataset (aka pt2)
part = MODEL_VERSIONS[-1]
dataset_name = base_dataset_name+part

parameter_dict = {  'dataset': dataset_name+'.inter',
                    'use_gpu':USE_GPU,
                    ## Environment settings https://recbole.io/docs/user_guide/config/environment_settings.html
                    'seed':SEED,
                    'state':'ERROR',
                    'data_path': save_path,
                    'save_dataset':SAVE_DATASET, #(bool): Whether or not to save filtered dataset. If True, save filtered dataset, otherwise it will not be saved. Defaults to False
                    'checkpoint_dir':save_path+base_dataset_name,
                    'show_progress': SHOW_PROGRESS,
                    'shuffle': SHUFFLE,
                    ## Data settings https://recbole.io/docs/user_guide/config/data_settings.html
                    'load_col': {'inter': ['user_id', 'item_id', 'timestamp']},
                    # 'user_inter_num_interval':'[1,inf)',
                    # 'benchmark_filename': BENCHMARK_FILENAMES,
                    
                    ## Training settings https://recbole.io/docs/user_guide/config/training_settings.html
                    # 'train_neg_sample_args': TRAIN_NEG_SAMPLE_ARGS,
                    
                    ## Evaluation settings https://recbole.io/docs/user_guide/config/evaluation_settings.html
                    'eval_args': EVAL_ARGS,
                    'metrics': METRICS, 
                    'topk':Ks,
                    'valid_metric':VALID_METRIC          
                    }

# note: timestamps are not converted to internal ids, but need to be set to the correct dtype
_, _, internal_dataset_pt2,\
        internal_train_pt2,\
            internal_valid_pt2,\
                internal_test_pt2 = setup_config_and_dataset(model_name, dataset_name, parameter_dict)

external_valid_pt2 = recbole_dataset_2_external_id_df(internal_valid_pt2.dataset, data_types_dict)
save_dataframe_2_atomic_file(df=external_valid_pt2,
                             save_path=save_path,
                             base_filename=base_filename,
                             specs_str=save_specs_str+part,
                             benchmark_filename=BENCHMARK_FILENAMES[1])

external_test_pt2 = recbole_dataset_2_external_id_df(internal_test_pt2.dataset, data_types_dict)
save_dataframe_2_atomic_file(df=external_test_pt2,
                             save_path=save_path,
                             base_filename=base_filename,
                             specs_str=save_specs_str+part,
                             benchmark_filename=BENCHMARK_FILENAMES[-1])

c:\Users\mjlav\anaconda3\envs\algorithms_transparency\lib\site-packages\recbole\data\dataset\dataset.py:648: FutureWarning:

A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.



c:\Users\mjlav\anaconda3\envs\algorithms_transparency\lib\site-packages\recbole\data\dataset\dataset.py:650: FutureWarning:

A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting value

Folder created:  processed_datasets/natural_data/palco2010/two_intervals/more_2interQ_df_PT_RD.50_pt2/
Dataset saved at processed_datasets/natural_data/palco2010/two_intervals/more_2interQ_df_PT_RD.50_pt2/, named more_2interQ_df_PT_RD.50_pt2.valid, in .csv and .inter
Dataset saved at processed_datasets/natural_data/palco2010/two_intervals/more_2interQ_df_PT_RD.50_pt2/, named more_2interQ_df_PT_RD.50_pt2.test, in .csv and .inter


## remove test sets from pt2's train, save test sets, save cleaned train sets

In [6]:
MODEL_VERSIONS[:-1]

['_pt1']

In [7]:
# remove all test sets from pt2's train set

parts_timestamps = {}
external_test_all_pts = pd.DataFrame()
for pti in MODEL_VERSIONS[:-1]: # all but pt2
    
    dataset_name = base_dataset_name+pti
    parameter_dict = {  'dataset': dataset_name+'.inter',
                        'use_gpu':USE_GPU,
                        ## Environment settings https://recbole.io/docs/user_guide/config/environment_settings.html
                        'seed':SEED,
                        'state':'ERROR',
                        'data_path': save_path,
                        'save_dataset':SAVE_DATASET, #(bool): Whether or not to save filtered dataset. If True, save filtered dataset, otherwise it will not be saved. Defaults to False
                        'checkpoint_dir':save_path+base_dataset_name,
                        'show_progress': SHOW_PROGRESS,
                        'shuffle': SHUFFLE,
                        ## Data settings https://recbole.io/docs/user_guide/config/data_settings.html
                        'load_col': {'inter': ['user_id', 'item_id', 'timestamp']},
                        # 'user_inter_num_interval':'[1,inf)',
                        # 'benchmark_filename': BENCHMARK_FILENAMES,
                        
                        ## Training settings https://recbole.io/docs/user_guide/config/training_settings.html
                        # 'train_neg_sample_args': TRAIN_NEG_SAMPLE_ARGS,
                        
                        ## Evaluation settings https://recbole.io/docs/user_guide/config/evaluation_settings.html
                        'eval_args': EVAL_ARGS,
                        'metrics': METRICS, 
                        'topk':Ks,
                        'valid_metric':VALID_METRIC          
                    }

    # note: timestamps are not converted to internal ids, but need to be set to the correct dtype
    _, _, internal_dataset_pti,\
            internal_train_pti,\
                internal_valid_pti,\
                    internal_test_pti = setup_config_and_dataset(model_name, dataset_name, parameter_dict)
    

    external_valid_pti = recbole_dataset_2_external_id_df(internal_valid_pti.dataset, data_types_dict)
    save_dataframe_2_atomic_file(df=external_valid_pti,
                                save_path=save_path,
                                base_filename=base_filename,
                                specs_str=save_specs_str+pti,
                                benchmark_filename=BENCHMARK_FILENAMES[1])

    external_test_pti = recbole_dataset_2_external_id_df(internal_test_pti.dataset, data_types_dict)
    save_dataframe_2_atomic_file(df=external_test_pti,
                                save_path=save_path,
                                base_filename=base_filename,
                                specs_str=save_specs_str+pti,
                                benchmark_filename=BENCHMARK_FILENAMES[-1])

    external_test_all_pts = pd.concat([external_test_all_pts, external_test_pti])



    ts = internal_train_pti.dataset.inter_feat.interaction[internal_train_pti.dataset.time_field].numpy().astype('float32')
    parts_timestamps[pti] = [ts.min(), ts.max()]


# remove from pt2's trainset and save it cleaned    
external_train_pt2 = recbole_dataset_2_external_id_df(internal_train_pt2.dataset, data_types_dict)

is_test_indicator_df = external_train_pt2.merge(external_test_all_pts, on=['user_id', 'item_id', 'timestamp'], how='left', indicator=True)
external_train_pt2_NoTest = is_test_indicator_df[is_test_indicator_df['_merge'] == 'left_only'].drop(columns=['_merge'])

# add pretrain to pt2's train set
ext_train_pt2_NT_PT = pd.concat([pretrain, external_train_pt2_NoTest])

save_dataframe_2_atomic_file(df=ext_train_pt2_NT_PT,
                                save_path=save_path,
                                base_filename=base_filename,
                                specs_str=save_specs_str+part,
                                benchmark_filename=BENCHMARK_FILENAMES[0])

# split pt2's cleaned-trainset in the different model parts and save it as the respective cleaned trainsets
for pti in MODEL_VERSIONS[:-1]:
    s, e = parts_timestamps[pti]
    external_train_pti_NoTest = external_train_pt2_NoTest.loc[(external_train_pt2_NoTest.timestamp>=s) & (external_train_pt2_NoTest.timestamp<=e),:]
    print('is there test interactions in both test and train sets?',pti, external_train_pti_NoTest.merge(external_test_all_pts, on=['user_id', 'item_id', 'timestamp'], how='left', indicator=True)._merge.value_counts())

    ext_train_pti_NT_PT = pd.concat([pretrain, external_train_pti_NoTest])

    save_dataframe_2_atomic_file(df=ext_train_pti_NT_PT,
                                save_path=save_path,
                                base_filename=base_filename,
                                specs_str=save_specs_str+pti,
                                benchmark_filename=BENCHMARK_FILENAMES[0])



c:\Users\mjlav\anaconda3\envs\algorithms_transparency\lib\site-packages\recbole\data\dataset\dataset.py:648: FutureWarning:

A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.



c:\Users\mjlav\anaconda3\envs\algorithms_transparency\lib\site-packages\recbole\data\dataset\dataset.py:650: FutureWarning:

A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting value

Folder created:  processed_datasets/natural_data/palco2010/two_intervals/more_2interQ_df_PT_RD.50_pt1/
Dataset saved at processed_datasets/natural_data/palco2010/two_intervals/more_2interQ_df_PT_RD.50_pt1/, named more_2interQ_df_PT_RD.50_pt1.valid, in .csv and .inter
Dataset saved at processed_datasets/natural_data/palco2010/two_intervals/more_2interQ_df_PT_RD.50_pt1/, named more_2interQ_df_PT_RD.50_pt1.test, in .csv and .inter
Dataset saved at processed_datasets/natural_data/palco2010/two_intervals/more_2interQ_df_PT_RD.50_pt2/, named more_2interQ_df_PT_RD.50_pt2.train, in .csv and .inter
is there test interactions in both test and train sets? _pt1 _merge
left_only     428650
right_only         0
both               0
Name: count, dtype: int64
Dataset saved at processed_datasets/natural_data/palco2010/two_intervals/more_2interQ_df_PT_RD.50_pt1/, named more_2interQ_df_PT_RD.50_pt1.train, in .csv and .inter


# No Drift

## load pt2, save pt2's valid and test

In [8]:
model_name = 'BPR' # for the sake of having one, BPR was chosen, but any other yields the same results in theory


save_path, base_filename, specs_str = ('processed_datasets/natural_data/palco2010/two_intervals/',
                                        'more_2interQ_df', 
                                        'NPT_ND') 
save_specs_str =  'PT_ND'
BENCHMARK_FILENAMES = ['train', 'valid', 'test']

base_dataset_name = base_filename+'_'+specs_str


# from complete dataset (aka pt2)
part = MODEL_VERSIONS[-1]
dataset_name = base_dataset_name+part

parameter_dict = {  'dataset': dataset_name+'.inter',
                    'use_gpu':USE_GPU,
                    ## Environment settings https://recbole.io/docs/user_guide/config/environment_settings.html
                    'seed':SEED,
                    'state':'ERROR',
                    'data_path': save_path,
                    'save_dataset':SAVE_DATASET, #(bool): Whether or not to save filtered dataset. If True, save filtered dataset, otherwise it will not be saved. Defaults to False
                    'checkpoint_dir':save_path+base_dataset_name,
                    'show_progress': SHOW_PROGRESS,
                    'shuffle': SHUFFLE,
                    ## Data settings https://recbole.io/docs/user_guide/config/data_settings.html
                    'load_col': {'inter': ['user_id', 'item_id', 'timestamp']},
                    # 'user_inter_num_interval':'[1,inf)',
                    # 'benchmark_filename': BENCHMARK_FILENAMES,
                    
                    ## Training settings https://recbole.io/docs/user_guide/config/training_settings.html
                    # 'train_neg_sample_args': TRAIN_NEG_SAMPLE_ARGS,
                    
                    ## Evaluation settings https://recbole.io/docs/user_guide/config/evaluation_settings.html
                    'eval_args': EVAL_ARGS,
                    'metrics': METRICS, 
                    'topk':Ks,
                    'valid_metric':VALID_METRIC          
                    }

# note: timestamps are not converted to internal ids, but need to be set to the correct dtype
_, _, internal_dataset_pt2,\
        internal_train_pt2,\
            internal_valid_pt2,\
                internal_test_pt2 = setup_config_and_dataset(model_name, dataset_name, parameter_dict)

external_valid_pt2 = recbole_dataset_2_external_id_df(internal_valid_pt2.dataset, data_types_dict)
save_dataframe_2_atomic_file(df=external_valid_pt2,
                             save_path=save_path,
                             base_filename=base_filename,
                             specs_str=save_specs_str+part,
                             benchmark_filename=BENCHMARK_FILENAMES[1])

external_test_pt2 = recbole_dataset_2_external_id_df(internal_test_pt2.dataset, data_types_dict)
save_dataframe_2_atomic_file(df=external_test_pt2,
                             save_path=save_path,
                             base_filename=base_filename,
                             specs_str=save_specs_str+part,
                             benchmark_filename=BENCHMARK_FILENAMES[-1])

c:\Users\mjlav\anaconda3\envs\algorithms_transparency\lib\site-packages\recbole\data\dataset\dataset.py:648: FutureWarning:

A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.



c:\Users\mjlav\anaconda3\envs\algorithms_transparency\lib\site-packages\recbole\data\dataset\dataset.py:650: FutureWarning:

A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting value

Folder created:  processed_datasets/natural_data/palco2010/two_intervals/more_2interQ_df_PT_ND_pt2/
Dataset saved at processed_datasets/natural_data/palco2010/two_intervals/more_2interQ_df_PT_ND_pt2/, named more_2interQ_df_PT_ND_pt2.valid, in .csv and .inter
Dataset saved at processed_datasets/natural_data/palco2010/two_intervals/more_2interQ_df_PT_ND_pt2/, named more_2interQ_df_PT_ND_pt2.test, in .csv and .inter


## remove test sets from pt2's train, save test sets, save cleaned train sets

In [9]:
MODEL_VERSIONS[:-1]

['_pt1']

In [10]:
# remove all test sets from pt2's train set

parts_timestamps = {}
external_test_all_pts = pd.DataFrame()
for pti in MODEL_VERSIONS[:-1]: # all but pt2
    
    dataset_name = base_dataset_name+pti
    parameter_dict = {  'dataset': dataset_name+'.inter',
                        'use_gpu':USE_GPU,
                        ## Environment settings https://recbole.io/docs/user_guide/config/environment_settings.html
                        'seed':SEED,
                        'state':'ERROR',
                        'data_path': save_path,
                        'save_dataset':SAVE_DATASET, #(bool): Whether or not to save filtered dataset. If True, save filtered dataset, otherwise it will not be saved. Defaults to False
                        'checkpoint_dir':save_path+base_dataset_name,
                        'show_progress': SHOW_PROGRESS,
                        'shuffle': SHUFFLE,
                        ## Data settings https://recbole.io/docs/user_guide/config/data_settings.html
                        'load_col': {'inter': ['user_id', 'item_id', 'timestamp']},
                        # 'user_inter_num_interval':'[1,inf)',
                        # 'benchmark_filename': BENCHMARK_FILENAMES,
                        
                        ## Training settings https://recbole.io/docs/user_guide/config/training_settings.html
                        # 'train_neg_sample_args': TRAIN_NEG_SAMPLE_ARGS,
                        
                        ## Evaluation settings https://recbole.io/docs/user_guide/config/evaluation_settings.html
                        'eval_args': EVAL_ARGS,
                        'metrics': METRICS, 
                        'topk':Ks,
                        'valid_metric':VALID_METRIC          
                    }

    # note: timestamps are not converted to internal ids, but need to be set to the correct dtype
    _, _, internal_dataset_pti,\
            internal_train_pti,\
                internal_valid_pti,\
                    internal_test_pti = setup_config_and_dataset(model_name, dataset_name, parameter_dict)
    

    external_valid_pti = recbole_dataset_2_external_id_df(internal_valid_pti.dataset, data_types_dict)
    save_dataframe_2_atomic_file(df=external_valid_pti,
                                save_path=save_path,
                                base_filename=base_filename,
                                specs_str=save_specs_str+pti,
                                benchmark_filename=BENCHMARK_FILENAMES[1])

    external_test_pti = recbole_dataset_2_external_id_df(internal_test_pti.dataset, data_types_dict)
    save_dataframe_2_atomic_file(df=external_test_pti,
                                save_path=save_path,
                                base_filename=base_filename,
                                specs_str=save_specs_str+pti,
                                benchmark_filename=BENCHMARK_FILENAMES[-1])

    external_test_all_pts = pd.concat([external_test_all_pts, external_test_pti])



    ts = internal_train_pti.dataset.inter_feat.interaction[internal_train_pti.dataset.time_field].numpy().astype('float32')
    parts_timestamps[pti] = [ts.min(), ts.max()]


# remove from pt2's trainset and save it cleaned    
external_train_pt2 = recbole_dataset_2_external_id_df(internal_train_pt2.dataset, data_types_dict)

is_test_indicator_df = external_train_pt2.merge(external_test_all_pts, on=['user_id', 'item_id', 'timestamp'], how='left', indicator=True)
external_train_pt2_NoTest = is_test_indicator_df[is_test_indicator_df['_merge'] == 'left_only'].drop(columns=['_merge'])

# add pretrain to pt2's train set
ext_train_pt2_NT_PT = pd.concat([pretrain, external_train_pt2_NoTest])

save_dataframe_2_atomic_file(df=ext_train_pt2_NT_PT,
                                save_path=save_path,
                                base_filename=base_filename,
                                specs_str=save_specs_str+part,
                                benchmark_filename=BENCHMARK_FILENAMES[0])

# split pt2's cleaned-trainset in the different model parts and save it as the respective cleaned trainsets
for pti in MODEL_VERSIONS[:-1]:
    s, e = parts_timestamps[pti]
    external_train_pti_NoTest = external_train_pt2_NoTest.loc[(external_train_pt2_NoTest.timestamp>=s) & (external_train_pt2_NoTest.timestamp<=e),:]
    print('is there test interactions in both test and train sets?',pti, external_train_pti_NoTest.merge(external_test_all_pts, on=['user_id', 'item_id', 'timestamp'], how='left', indicator=True)._merge.value_counts())

    ext_train_pti_NT_PT = pd.concat([pretrain, external_train_pti_NoTest])

    save_dataframe_2_atomic_file(df=ext_train_pti_NT_PT,
                                save_path=save_path,
                                base_filename=base_filename,
                                specs_str=save_specs_str+pti,
                                benchmark_filename=BENCHMARK_FILENAMES[0])

c:\Users\mjlav\anaconda3\envs\algorithms_transparency\lib\site-packages\recbole\data\dataset\dataset.py:648: FutureWarning:

A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.



c:\Users\mjlav\anaconda3\envs\algorithms_transparency\lib\site-packages\recbole\data\dataset\dataset.py:650: FutureWarning:

A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting value

Folder created:  processed_datasets/natural_data/palco2010/two_intervals/more_2interQ_df_PT_ND_pt1/
Dataset saved at processed_datasets/natural_data/palco2010/two_intervals/more_2interQ_df_PT_ND_pt1/, named more_2interQ_df_PT_ND_pt1.valid, in .csv and .inter
Dataset saved at processed_datasets/natural_data/palco2010/two_intervals/more_2interQ_df_PT_ND_pt1/, named more_2interQ_df_PT_ND_pt1.test, in .csv and .inter
Dataset saved at processed_datasets/natural_data/palco2010/two_intervals/more_2interQ_df_PT_ND_pt2/, named more_2interQ_df_PT_ND_pt2.train, in .csv and .inter
is there test interactions in both test and train sets? _pt1 _merge
left_only     428650
right_only         0
both               0
Name: count, dtype: int64
Dataset saved at processed_datasets/natural_data/palco2010/two_intervals/more_2interQ_df_PT_ND_pt1/, named more_2interQ_df_PT_ND_pt1.train, in .csv and .inter
